# A3 — Self-Tooling

**Delegation loop where the manager can authorize workers to create tools at runtime.**

**Fictional task**: invent a scoring tool and rank five fictional product reviews.

In [ ]:
# --- Load API key from the canonical env file (see memory `reference_api_keys`) ---
import os
from pathlib import Path

env_file = Path('/home/shumway/projects/meta-agents/.env')
if env_file.exists() and not os.environ.get('OPENROUTER_API_KEY'):
    for raw in env_file.read_text().splitlines():
        s = raw.strip()
        if s.startswith('OPENROUTER_API_KEY='):
            os.environ['OPENROUTER_API_KEY'] = s.split('=', 1)[1].strip().strip('"').strip("'")
            break

assert os.environ.get('OPENROUTER_API_KEY'), 'OPENROUTER_API_KEY missing'
# Default worker model for agents that do not declare their own (DAG engine consults LLM_MODEL).
os.environ.setdefault('LLM_MODEL', 'deepseek/deepseek-chat-v3.1')
print('OpenRouter key loaded. Default model:', os.environ['LLM_MODEL'])

## Load + A3 compliance

In [ ]:
from pathlib import Path
from awp.parser import parse_manifest, parse_agent
from awp.validator import check_compliance, AutonomyLevel

WORKFLOW_DIR = Path('/home/shumway/projects/agent-workflow-protocol/examples/workflows/10-skill-and-tool-generation')
manifest = parse_manifest(WORKFLOW_DIR / 'workflow.awp.yaml')
agents = {}
for ad in (WORKFLOW_DIR / 'agents').iterdir():
    a = ad / 'agent.awp.yaml'
    if a.exists():
        agents[ad.name] = parse_agent(a)

result = check_compliance(manifest, agents, target_level=AutonomyLevel.A3_SELF_TOOLING)
assert result.level >= AutonomyLevel.A3_SELF_TOOLING, f'Not A3: {result.errors}'
print(f'A3 compliant. Manager may authorize tool creation.')

## Run with tool-creation enabled

In [ ]:
import json
import logging
logging.basicConfig(level=logging.WARNING)

from awp.runtime import WorkflowRunner

TASK = (
    'Rank these five fictional product reviews from best to worst by inventing a small scoring helper tool. '
    'Reviews: '
    '(1) "Battery lasted forever — loved it.", '
    '(2) "Broke after two days. Waste of money.", '
    '(3) "Decent, but the app is buggy.", '
    '(4) "Perfect — would buy again.", '
    '(5) "Confusing setup, but works ok once configured." '
    'Return an ordered list with your score rationale per review.'
)
runner = WorkflowRunner(
    WORKFLOW_DIR,
    manager_model='openai/gpt-5-mini',
    worker_model='deepseek/deepseek-chat-v3.1',
)
result = runner.run(TASK)

print(json.dumps({k: v for k, v in result.items() if not k.startswith('_')}, indent=2, default=str)[:3000])

## Assertions (E2E rubric)

In [ ]:
assert isinstance(result, dict) and result, 'empty result'
has_content = any(
    (isinstance(v, (str, dict, list)) and bool(v)) for k, v in result.items() if not k.startswith('_')
)
assert has_content, f'no content in result. Keys: {list(result)}'
tool_dir = WORKFLOW_DIR / 'shared' / 'dynamic_tools'
if tool_dir.exists():
    created = list(tool_dir.glob('*.json'))
    print(f'Dynamic tools in registry: {len(created)}')
print('A3 OK — self-tooling delegation loop closed.')